In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from collections import defaultdict
import time

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [30]:
class PrunableLinear(nn.Module):
    """
    Linear layer where every weight has a paired learnable gate.

    Forward
    -------
        gates          = sigmoid(gate_scores)   # in (0,1), same shape as weight
        pruned_weights = weight * gates          # element-wise hard mask
        output         = F.linear(x, pruned_weights, bias)

    Design choices
    --------------
    * gate_scores init = 0.0  →  sigmoid(0) = 0.5  (neutral; equally easy
      to push up or down, unlike init=2 which starts gates at 0.88 and
      requires a lot of gradient work to close them).
    * Both weight and gate_scores are registered nn.Parameters so the
      optimiser updates them jointly; gradients flow through sigmoid and
      the element-wise product without any custom backward needed.
    """

    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features)) if bias else None

        # Init gate_scores = 0  →  all gates start at 0.5
        self.gate_scores = nn.Parameter(
            torch.full((out_features, in_features), 0.9)
          )

        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / np.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates = torch.clamp(self.gate_scores, 0.0, 1.0)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    # Helpers
    def get_gates(self) -> torch.Tensor:
        """Detached gate values for logging/analysis."""
        return torch.sigmoid(self.gate_scores).detach()

    def gate_mean(self) -> float:
        return self.get_gates().mean().item()

    def num_gates(self) -> int:
        return self.gate_scores.numel()

    def sparsity(self, threshold: float = 1e-2) -> float:
        return (self.get_gates() < threshold).float().mean().item()

    def extra_repr(self):
        return (f"in={self.in_features}, out={self.out_features}, "
                f"gates={self.num_gates():,}")

In [31]:
class SelfPruningNet(nn.Module):
    """
    CIFAR-10 feed-forward network: 3072 -> 512 -> 256 -> 128 -> 10
    All linear layers are PrunableLinear.
    """

    def __init__(self, hidden_dims=(512, 256, 128)):
        super().__init__()
        in_dim = 3 * 32 * 32
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [PrunableLinear(prev, h),
                       nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True),
                       nn.Dropout(0.2)]
            prev = h
        layers.append(PrunableLinear(prev, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

    # Gate / sparsity utilities
    def prunable_layers(self):
        return [m for m in self.modules() if isinstance(m, PrunableLinear)]

    def sparsity_loss(self) -> torch.Tensor:
        """
        Normalised L1 on gates  →  value always in (0, 1).

        SparsityLoss = mean_over_all_gates( sigmoid(gate_score) )

        Dividing by N_gates makes lambda scale-independent of network size:
          lambda = 0.1  means the optimizer treats closing 10% more gates
                        as worth the same as reducing cross-entropy by 0.1.
        """
        gate_sum    = torch.zeros(1, device=next(self.parameters()).device)
        total_gates = 0
        for layer in self.prunable_layers():
            gate_sum    = gate_sum + torch.sigmoid(layer.gate_scores).sum()
            total_gates += layer.num_gates()
        return gate_sum / total_gates       # scalar in (0, 1)

    def overall_sparsity(self, threshold=1e-2) -> float:
        pruned = total = 0
        for l in self.prunable_layers():
            g = l.get_gates(); total += g.numel()
            pruned += (g < threshold).sum().item()
        return 100.0 * pruned / total if total else 0.0

    def gate_mean(self) -> float:
        vals = [l.gate_mean() for l in self.prunable_layers()]
        return float(np.mean(vals))

    def all_gates(self) -> np.ndarray:
        return np.concatenate(
            [l.get_gates().cpu().numpy().ravel() for l in self.prunable_layers()])

In [32]:
def get_loaders(batch_size=128, num_workers=2):
    mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    test_tf  = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    tr = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    te = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    return (DataLoader(tr, batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True),
            DataLoader(te, 256,        shuffle=False, num_workers=num_workers, pin_memory=True))

In [33]:
def make_optimizer(model, weight_lr=1e-3, gate_lr=0.1, wd=1e-4):
    """
    Two parameter groups:
      • weights / biases / BN params  →  lr = weight_lr  (default 0.001)
      • gate_scores                   →  lr = gate_lr    (default 0.01)

    Gates get a 10× higher learning rate so they respond quickly to the
    sparsity gradient without destabilising the weight optimisation.
    """
    gate_params   = [p for n, p in model.named_parameters() if "gate_scores" in n]
    other_params  = [p for n, p in model.named_parameters() if "gate_scores" not in n]
    return optim.Adam([
        {"params": other_params, "lr": weight_lr, "weight_decay": wd},
        {"params": gate_params,  "lr": gate_lr,   "weight_decay": 0.0},
    ])

In [36]:
def train_one_epoch(model, loader, optimizer, lam, device, epoch):
  freeze_gates = (epoch < 5)
  for m in model.prunable_layers():
    m.gate_scores.requires_grad_(not freeze_gates)

  model.train()
  cls_sum = spar_sum = correct = seen = 0
  for imgs, labels in loader:
      imgs, labels = imgs.to(device), labels.to(device)
      optimizer.zero_grad()
      logits    = model(imgs)
      cls_loss  = F.cross_entropy(logits, labels)
      spar_loss = model.sparsity_loss()
      (cls_loss + lam * spar_loss).backward()
      nn.utils.clip_grad_norm_(model.parameters(), 5.0)
      optimizer.step()
      n = imgs.size(0)
      cls_sum  += cls_loss.item()  * n
      spar_sum += spar_loss.item() * n
      correct  += (logits.argmax(1) == labels).sum().item()
      seen     += n
  N = len(loader.dataset)
  return {"cls": cls_sum/N, "spar": spar_sum/N, "acc": 100*correct/seen}


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    loss_sum = correct = seen = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits    = model(imgs)
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        correct  += (logits.argmax(1) == labels).sum().item()
        seen     += imgs.size(0)
    return {"loss": loss_sum/seen, "acc": 100*correct/seen}


def run_experiment(lam, epochs=40, weight_lr=1e-3, gate_lr=1e-2,
                   batch_size=128, hidden_dims=(512,256,128), verbose=True):
    train_loader, test_loader = get_loaders(batch_size)
    model     = SelfPruningNet(hidden_dims).to(DEVICE)
    optimizer = make_optimizer(model, weight_lr, gate_lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = defaultdict(list)

    if verbose:
        ng = sum(l.num_gates() for l in model.prunable_layers())
        print(f"\n{'='*70}")
        print(f"  lambda={lam}  |  gates={ng:,}  |  gate_lr={gate_lr}  |  {epochs} epochs")
        print(f"{'='*70}")
        print(f"  {'Ep':>3}  {'CLS':>7}  {'SPAR':>7}  {'GateMean':>9}  "
              f"{'TrainAcc':>9}  {'TestAcc':>8}  {'Sparse':>7}  {'Time':>6}")
        print(f"  {'-'*65}")

    for ep in range(1, epochs+1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, optimizer, lam, DEVICE, ep)
        te = evaluate(model, test_loader, DEVICE)
        scheduler.step()
        sp = model.overall_sparsity()
        gm = model.gate_mean()

        history["cls"].append(tr["cls"])
        history["spar"].append(tr["spar"])
        history["train_acc"].append(tr["acc"])
        history["test_acc"].append(te["acc"])
        history["sparsity"].append(sp)
        history["gate_mean"].append(gm)

        if verbose and (ep % 5 == 0 or ep == 1):
            print(f"  {ep:3d}  {tr['cls']:7.4f}  {tr['spar']:7.4f}  "
                  f"{gm:9.4f}  {tr['acc']:8.1f}%  {te['acc']:7.1f}%  "
                  f"{sp:6.1f}%  [{time.time()-t0:.1f}s]")

    final = evaluate(model, test_loader, DEVICE)
    sp    = model.overall_sparsity()
    if verbose:
        print(f"\n  => Final Test Accuracy : {final['acc']:.2f}%")
        print(f"  => Final Sparsity      : {sp:.2f}%")
        print(f"  => Final Gate Mean     : {model.gate_mean():.4f}")

    return {"lam": lam, "model": model, "history": dict(history),
            "test_acc": final["acc"], "sparsity": sp}

In [37]:
LAMBDAS = [1.0, 3.0, 10.0]
EPOCHS  = 40

results = {}
for lam in LAMBDAS:
    results[lam] = run_experiment(lam=lam, epochs=EPOCHS, verbose=True)


  lambda=1.0  |  gates=1,737,984  |  gate_lr=0.01  |  40 epochs
   Ep      CLS     SPAR   GateMean   TrainAcc   TestAcc   Sparse    Time
  -----------------------------------------------------------------
    1   1.8500   0.7109     0.7109      32.8%     42.0%     0.0%  [20.4s]
    5   1.5640   0.6789     0.6893      43.8%     49.1%     0.0%  [20.7s]
   10   1.4435   0.4284     0.6010      48.1%     53.5%     1.4%  [20.7s]
   15   1.3815   0.3033     0.5447      50.2%     56.4%     9.6%  [20.3s]
   20   1.3361   0.2336     0.5040      51.8%     57.3%    20.4%  [19.7s]
   25   1.3011   0.1936     0.4738      53.1%     58.2%    31.9%  [20.2s]
   30   1.2700   0.1712     0.4531      54.5%     59.0%    41.4%  [21.1s]
   35   1.2531   0.1599     0.4415      55.1%     59.7%    46.9%  [20.4s]
   40   1.2440   0.1567     0.4383      55.4%     59.7%    48.3%  [19.5s]

  => Final Test Accuracy : 59.67%
  => Final Sparsity      : 48.25%
  => Final Gate Mean     : 0.4383

  lambda=3.0  |  gates=1

In [38]:
print("\n" + "="*60)
print(f"  {'Lambda':<8}  {'Test Accuracy':>14}  {'Sparsity (%)':>13}")
print("-"*60)
for lam, tag in zip(LAMBDAS, ["(low)", "(medium)", "(high)"]):
    r = results[lam]
    print(f"  {lam:<8}  {r['test_acc']:>13.2f}%  {r['sparsity']:>12.2f}%  {tag}")
print("="*60)


  Lambda     Test Accuracy   Sparsity (%)
------------------------------------------------------------
  1.0               59.67%         48.25%  (low)
  3.0               59.72%         72.25%  (medium)
  10.0              58.97%         87.89%  (high)


In [41]:
COLORS = ["#1565C0", "#E65100", "#B71C1C"]
NAMES  = {1.0: "λ=1.0 (low)", 3.0: "λ=3.0 (med)", 10.0: "λ=10.0 (high)"}

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35)

# Row 1: Gate histograms
for i, lam in enumerate(LAMBDAS):
    ax    = fig.add_subplot(gs[0, i])
    gates = results[lam]["model"].all_gates()
    pct   = (gates < 0.01).mean() * 100

    # Pruned gates in red, active in colour
    ax.hist(gates[gates <  0.01], bins=40, color="#E53935", alpha=0.90,
            label=f"Pruned  (<0.01):  {pct:.1f}%")
    ax.hist(gates[gates >= 0.01], bins=80, color=COLORS[i], alpha=0.70,
            label=f"Active  (>=0.01): {100-pct:.1f}%")
    ax.axvline(0.01, color="black", linestyle="--", lw=1.2, label="threshold")
    ax.set_title(f"Gate Distribution\n{NAMES[lam]}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Gate value  σ(gate_score)", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.legend(fontsize=8)

# Row 2: Accuracy & Sparsity curves
ax_acc  = fig.add_subplot(gs[1, 0])
ax_spar = fig.add_subplot(gs[1, 1])
ax_gate = fig.add_subplot(gs[1, 2])

for i, lam in enumerate(LAMBDAS):
    h  = results[lam]["history"]
    ep = range(1, len(h["test_acc"]) + 1)
    ax_acc.plot( ep, h["test_acc"],  color=COLORS[i], lw=2, label=NAMES[lam])
    ax_spar.plot(ep, h["sparsity"],  color=COLORS[i], lw=2, label=NAMES[lam])
    ax_gate.plot(ep, h["gate_mean"], color=COLORS[i], lw=2, label=NAMES[lam])

for ax, title, yl in [
    (ax_acc,  "Test Accuracy vs Epochs",   "Test Accuracy (%)"),
    (ax_spar, "Sparsity vs Epochs",        "% Gates Pruned"),
    (ax_gate, "Mean Gate Value vs Epochs", "Mean σ(gate_score)"),
]:
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel(yl, fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

# Row 3: Sparsity vs Accuracy trade-off scatter
ax_trade = fig.add_subplot(gs[2, :])
for i, lam in enumerate(LAMBDAS):
    h  = results[lam]["history"]
    ax_trade.plot(h["sparsity"], h["test_acc"],
                  color=COLORS[i], lw=1.5, alpha=0.6)
    ax_trade.scatter(results[lam]["sparsity"], results[lam]["test_acc"],
                     color=COLORS[i], s=120, zorder=5,
                     label=f"{NAMES[lam]}  |  final: "
                           f"acc={results[lam]['test_acc']:.1f}%, "
                           f"sparse={results[lam]['sparsity']:.1f}%")

ax_trade.set_title("Accuracy–Sparsity Trade-off  (each point = 1 epoch)",
                   fontsize=11, fontweight="bold")
ax_trade.set_xlabel("Sparsity (%)", fontsize=10)
ax_trade.set_ylabel("Test Accuracy (%)", fontsize=10)
ax_trade.legend(fontsize=9); ax_trade.grid(alpha=0.25)

plt.suptitle("Self-Pruning Neural Network  –  CIFAR-10",
             fontsize=14, fontweight="bold", y=1.01)
plt.savefig("gate_distributions.png", dpi=150, bbox_inches="tight")
print("Plot saved -> gate_distributions.png")
plt.show()

Plot saved -> gate_distributions.png


In [42]:
rows_md = "\n".join(
    f"| {lam} | {results[lam]['test_acc']:.2f}% | {results[lam]['sparsity']:.2f}% |"
    for lam in LAMBDAS
)

report = f"""# Self-Pruning Neural Network – Report

## 1  Why L1 on Sigmoid Gates Encourages Sparsity

The training objective is:

```
Total Loss = CrossEntropyLoss(ŷ, y)
           + λ × mean_over_all_gates( sigmoid(gate_score_ij) )
```

**Why sigmoid?**  It squashes each raw `gate_score ∈ ℝ` into **(0, 1)**.
A value near 1 = weight fully active; near 0 = weight pruned.

**Why L1 (sum/mean) and not L2?**
The L1 sub-gradient is a *constant* ±1 regardless of the gate's magnitude.
This means the sparsity gradient never shrinks, even as a gate approaches zero —
it keeps pulling until the gate actually reaches zero.
L2 would produce a gradient proportional to the current value, so weights
asymptotically approach zero but never reach it (no true sparsity).
This is the same reason LASSO regression produces exact zeros while Ridge does not.

**Why normalise by N_gates?**
The raw sum over 1.7 M gates would be ~10⁶, dwarfing the classification loss.
Dividing by N keeps SparsityLoss ∈ (0, 1), so λ is directly interpretable:
λ = 0.5 means "I allow a 0.5-unit increase in cross-entropy in exchange for
closing one gate unit of capacity."

**The λ trade-off:**
| Lambda | Effect |
|--------|--------|
| 0.1 | Mild sparsity pressure; most gates remain active; best accuracy |
| 0.5 | Moderate; meaningful pruning with small accuracy cost |
| 1.0 | Aggressive; high sparsity; accuracy may drop |

---

## 2  Results Summary

| Lambda | Test Accuracy | Sparsity Level (%) |
|--------|:-------------:|:------------------:|
{rows_md}

---

## 3  Gate Distribution Plot

![Gate Distributions](gate_distributions.png)

A successful self-pruning run shows:
- A **large red spike near gate = 0** for pruned weights
- A **blue/orange cluster** of remaining active gates
- The red spike becomes dominant as λ increases

---

## 4  Implementation Notes

- **Separate LR for gates** (10× the weight LR) allows gate_scores to respond
  quickly to the sparsity gradient without destabilising weight training.
- **gate_scores init = 0** → sigmoid(0) = 0.5, a neutral starting point that
  is equally easy to push toward 0 or 1.
- **Gradient clipping** (max_norm=5) prevents the large sparsity gradients at
  high λ from causing instability early in training.
"""

with open("report.md", "w") as f:
    f.write(report)
print("Report saved -> report.md")
print(report)

Report saved -> report.md
# Self-Pruning Neural Network – Report
 
## 1  Why L1 on Sigmoid Gates Encourages Sparsity
 
The training objective is:
 
```
Total Loss = CrossEntropyLoss(ŷ, y)
           + λ × mean_over_all_gates( sigmoid(gate_score_ij) )
```
 
**Why sigmoid?**  It squashes each raw `gate_score ∈ ℝ` into **(0, 1)**.
A value near 1 = weight fully active; near 0 = weight pruned.
 
**Why L1 (sum/mean) and not L2?**
The L1 sub-gradient is a *constant* ±1 regardless of the gate's magnitude.
This means the sparsity gradient never shrinks, even as a gate approaches zero —
it keeps pulling until the gate actually reaches zero.
L2 would produce a gradient proportional to the current value, so weights
asymptotically approach zero but never reach it (no true sparsity).
This is the same reason LASSO regression produces exact zeros while Ridge does not.
 
**Why normalise by N_gates?**
The raw sum over 1.7 M gates would be ~10⁶, dwarfing the classification loss.
Dividing by N keeps Spars